# Structured Output with Function Calling

Getting structured data out of an LLM — a typed dict, a list of objects, or a nested model — is one of the most practical patterns in production LLM applications. Function calling is the most reliable mechanism: the LLM fills in a JSON schema instead of generating free text, and Pydantic validates the result.

**What you'll build:** an agent that returns typed Pydantic models for tasks like entity extraction, report generation, and multi-step research with a structured summary.

**Time:** ~20 min | **Difficulty:** Intermediate

**What you'll learn:**
- How to use a Pydantic model as a tool schema to force structured output
- The "extraction tool" pattern: define the desired schema as a tool, the LLM "calls" it to produce output
- How to validate and deserialize LLM output into typed Python objects
- Combining structured output with real tool calls in the same agent
- Handling optional fields and nested models

## Prerequisites

You need:
- An **OpenAI API key** (`OPENAI_API_KEY`)
- The `synapsekit` and `pydantic` packages (installed below)

In [ ]:
!pip install synapsekit pydantic -q

In [ ]:
import os

# Set your API key here
# os.environ['OPENAI_API_KEY'] = 'sk-...'

## Step 1: Define output schemas as Pydantic models

Define the target shape of the output using Pydantic. `Field(description=...)` strings become the parameter descriptions that guide the LLM's field population.

In [ ]:
import asyncio
import json
from typing import Any
from pydantic import BaseModel, Field
from synapsekit.agents import BaseTool, FunctionCallingAgent, ToolResult
from synapsekit.llms.openai import OpenAILLM


class CompanyProfile(BaseModel):
    name: str = Field(description="Official company name")
    industry: str = Field(description="Primary industry sector")
    founded_year: int | None = Field(None, description="Year the company was founded")
    headquarters: str = Field(description="City and country of headquarters")
    key_products: list[str] = Field(description="Top 3-5 products or services")
    competitors: list[str] = Field(description="Main competitors")
    summary: str = Field(description="2-3 sentence company summary")


class NewsArticle(BaseModel):
    title: str
    source: str
    published_date: str | None = None
    key_points: list[str] = Field(description="3-5 key takeaways from the article")
    sentiment: str = Field(description="Overall sentiment: positive, negative, or neutral")


class ResearchReport(BaseModel):
    topic: str
    findings: list[str] = Field(description="Key facts and findings, one per item")
    sources: list[str] = Field(description="Source names or URLs")
    confidence: str = Field(description="Confidence level: high, medium, or low")
    conclusion: str = Field(description="Single-paragraph conclusion")


print("Pydantic models defined.")
print(f"CompanyProfile fields: {list(CompanyProfile.model_fields.keys())}")

## Step 2: Create extraction tools from Pydantic models

The "extraction tool" pattern treats the Pydantic schema as a tool that the LLM "calls" to produce structured output. The tool's job is to receive the validated JSON, deserialize it, and store or return it.

In [ ]:
def make_extraction_tool(model_class: type[BaseModel], tool_name: str, description: str) -> BaseTool:
    """Create a BaseTool that extracts data into a Pydantic model."""

    # Convert Pydantic schema to OpenAI-compatible JSON Schema
    pydantic_schema = model_class.model_json_schema()

    class ExtractionTool(BaseTool):
        name = tool_name
        description = description
        parameters = pydantic_schema

        # Store the last extracted result for retrieval after the run
        last_result: model_class | None = None

        async def run(self, **kwargs: Any) -> ToolResult:
            try:
                # Pydantic validates and coerces the LLM's JSON output
                instance = model_class(**kwargs)
                ExtractionTool.last_result = instance
                return ToolResult(output=instance.model_dump_json())
            except Exception as e:
                return ToolResult(output="", error=f"Schema validation failed: {e}")

    return ExtractionTool()


# Test the factory
test_tool = make_extraction_tool(
    NewsArticle,
    tool_name="submit_article",
    description="Submit a news article analysis."
)
print(f"Created tool: {test_tool.name}")
print(f"Schema keys: {list(test_tool.parameters.get('properties', {}).keys())}")

## Step 3: Build a structured extraction agent

Wire the extraction tool into an agent with a system prompt that instructs the LLM to call the extraction tool rather than answering in plain text.

In [ ]:
company_extractor = make_extraction_tool(
    CompanyProfile,
    tool_name="extract_company_profile",
    description=(
        "Call this tool to return a structured company profile. "
        "Use it when you have gathered enough information to fill all fields."
    ),
)

agent = FunctionCallingAgent(
    llm=OpenAILLM(model="gpt-4o-mini"),
    tools=[company_extractor],
    system_prompt=(
        "You are a business analyst. When asked about a company, "
        "call extract_company_profile with all available information. "
        "Do not answer in plain text — always call the tool."
    ),
    max_iterations=3,
)

# Test extraction
asyncio.run(agent.run("Create a profile for Apple Inc."))
profile = company_extractor.__class__.last_result
if profile:
    print(f"Company: {profile.name}")
    print(f"Industry: {profile.industry}")
    print(f"Products: {profile.key_products[:3]}")

## Step 4: Combine real tools with structured output

For agents that need to search before extracting, combine research tools with the extraction tool.

In [ ]:
from synapsekit.agents import DuckDuckGoSearchTool, WikipediaTool

research_extractor = make_extraction_tool(
    ResearchReport,
    tool_name="submit_research_report",
    description=(
        "Call this tool AFTER completing research to submit the final structured report. "
        "Fill all fields based on information gathered from search and Wikipedia."
    ),
)

research_agent = FunctionCallingAgent(
    llm=OpenAILLM(model="gpt-4o-mini"),
    tools=[
        DuckDuckGoSearchTool(),
        WikipediaTool(),
        research_extractor,
    ],
    system_prompt=(
        "You are a research analyst. Research the given topic using search and Wikipedia, "
        "then submit a structured report using submit_research_report. "
        "Always call submit_research_report as your final action."
    ),
    max_iterations=8,
)

print("Research agent with 3 tools (search + wikipedia + report extractor) configured.")

## Step 5: Extract and validate the typed result

After `agent.run()`, recover the typed Pydantic object from the extraction tool and access it as a fully typed Python object.

In [ ]:
async def extract_company(company_name: str) -> CompanyProfile | None:
    await agent.run(f"Create a detailed profile for the company: {company_name}")
    return company_extractor.__class__.last_result

profile = asyncio.run(extract_company("OpenAI"))
if profile:
    print(f"Name: {profile.name}")
    print(f"Founded: {profile.founded_year}")
    print(f"Summary: {profile.summary[:100]}...")
    # Access as typed dict
    data = profile.model_dump()
    print(f"Keys: {list(data.keys())}")

## Complete Working Example

A self-contained script that researches two topics and returns typed `ResearchReport` objects, printing each report's structured fields.

In [ ]:
import asyncio
import json
from typing import Any
from pydantic import BaseModel, Field
from synapsekit.agents import (
    BaseTool,
    DuckDuckGoSearchTool,
    FunctionCallingAgent,
    ToolResult,
    WikipediaTool,
)
from synapsekit.llms.openai import OpenAILLM


class ResearchReport(BaseModel):
    topic: str = Field(description="Research topic")
    findings: list[str] = Field(description="3-5 key findings, one per list item")
    sources: list[str] = Field(description="Source names referenced")
    confidence: str = Field(description="Confidence level: high, medium, or low")
    conclusion: str = Field(description="One-paragraph conclusion")


def make_report_tool() -> BaseTool:
    class ReportTool(BaseTool):
        name = "submit_research_report"
        description = (
            "Submit the final structured research report. "
            "Call this as your last action after gathering all information."
        )
        parameters = ResearchReport.model_json_schema()
        last_result: ResearchReport | None = None

        async def run(self, **kwargs: Any) -> ToolResult:
            try:
                report = ResearchReport(**kwargs)
                ReportTool.last_result = report
                return ToolResult(output=f"Report submitted: {report.topic}")
            except Exception as e:
                return ToolResult(output="", error=str(e))

    return ReportTool()


async def research_topic(topic: str) -> ResearchReport | None:
    report_tool = make_report_tool()

    agent = FunctionCallingAgent(
        llm=OpenAILLM(model="gpt-4o-mini"),
        tools=[
            DuckDuckGoSearchTool(),
            WikipediaTool(),
            report_tool,
        ],
        system_prompt=(
            "You are a research analyst. Use search and Wikipedia to research the topic. "
            "Then call submit_research_report with structured findings. "
            "Always end by calling submit_research_report."
        ),
        max_iterations=8,
    )

    await agent.run(f"Research the following topic and submit a report: {topic}")
    return report_tool.__class__.last_result


async def main() -> None:
    topics = [
        "The current state of quantum computing hardware",
        "SynapseKit Python library for LLM applications",
    ]

    for topic in topics:
        print(f"\nResearching: {topic}")
        print("-" * 60)
        report = await research_topic(topic)

        if report is None:
            print("No report generated.")
            continue

        print(f"Topic:      {report.topic}")
        print(f"Confidence: {report.confidence}")
        print(f"\nFindings:")
        for i, finding in enumerate(report.findings, 1):
            print(f"  {i}. {finding}")
        print(f"\nSources: {', '.join(report.sources)}")
        print(f"\nConclusion:\n{report.conclusion}")

        # Access as a typed Python object — no manual JSON parsing needed
        report_dict = report.model_dump()
        print(f"\nJSON keys: {list(report_dict.keys())}")


asyncio.run(main())

## Next Steps

- [Multi-Tool Orchestration](https://synapsekit.github.io/synapsekit-docs/docs/guides/agents/multi-tool-orchestration) — combine structured output with a larger toolset
- [Tool Error Handling and Retries](https://synapsekit.github.io/synapsekit-docs/docs/guides/agents/tool-error-handling) — retry when Pydantic validation fails
- [SQL Database Agent](https://synapsekit.github.io/synapsekit-docs/docs/guides/agents/sql-database-agent) — return SQL query results as typed Pydantic models